# [SK 07.7.F - AI Foundry Agents with Bing using Declarative Spec](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
**Note**: `azure-ai-agents==1.1.0b4` and `azure-ai-projects==1.0.0` are automatically installed by semantic kernel 1.35.0.<br/>

Combining our ProductionLinePlugin with the Bing Grounding tool opens up a powerful hybrid use case: internal control + external awareness. This lets the agent not only manage our chocolate production lines but also make informed decisions based on real-time market trends, regulations, or competitor activity.<br/>
Note: the `BingConnectionId` is in the format of
`/subscriptions/sub_id/resourceGroups/rg/providers/Microsoft.MachineLearningServices/workspaces/workspace_id/connections/bing_connection_id`.

It can either be configured as an env var `AZURE_AI_AGENT_BING_CONNECTION_ID` or passed in as an extra to 
`create_from_yaml`: extras={"BingConnectionId": "<bing_connection_id>"}.<br/>

A minimal YAML declarative spec might look like the following:
```
type: foundry_agent
name: sk_aifoundry_agent-PYTHON-from-specs
instructions: You are a clever agent
description: This agent answers questions
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: ProductionLinePlugin.get_lines
    type: function
  - id: ProductionLinePlugin.update_line_status
    type: function
  - type: bing_grounding
    options:
      tool_connections:
        - ${AzureAI:BingConnectionId}
```

# Constants and Libraries

Just if needed, log on Azure
```
import os

# Login with tenant ID
os.system("az login --tenant 3ad0b905-34ab-4116-93d9-c1dcc2d35af6 --output none") # --use-device-code

# Set the subscription programmatically
os.system("az account set --subscription eca2eddb-0f0c-4351-a634-52751499eeea")
```

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-chocolate-lines"

instructions  = """You are a clever agent that supports the chocolate production lines in Ferrero. You have full access to Internet. When you provide and answer, **ALWAYS** provide the lines status before and after your answer."""

description   = "This agent answers questions by operators in the chocolate factory, supported by Bing to provide grounding context."""

project_endpoint = os.environ["AIF_STD_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  "gpt-4.1" #os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# Create AI FOUNDRY PROJECT CLIENT using Semantic Kernel SDK

In [2]:
from semantic_kernel.agents import AzureAIAgent, AzureAIAgentSettings
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

credential = DefaultAzureCredential()

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())
agent_settings = AzureAIAgentSettings() # other than "from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings"
agent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4.1', endpoint='https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None)

# Native Plugin

In [3]:
class ProductionLinePlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function

    def __init__(self):
        self.lines = [
            {"id": 0, "name": "Rocher Line", "status": "stopped"},
            {"id": 1, "name": "Mon Chéri Line", "status": "stopped"},
            {"id": 2, "name": "Kinder Bueno Line", "status": "stopped"},
        ]

    @kernel_function(
        name="get_lines",
        description="Returns the current status of all production lines",
    )
    def get_status(
        self,
    ) -> Annotated[str, "List of production lines and their status"]:
        return str(self.lines)

    @kernel_function(
        name="update_line_status",
        description="Starts or stops a specific production line",
    )
    def update_status(
        self,
        id: int,
        status: str,
    ) -> Annotated[str, "Updated line status"]:
        for line in self.lines:
            if line["id"] == id:
                line["status"] = status
                return str(line)
        return "Line not found"

# Create an AI Foundry Agent

In [4]:
connections = project_client.connections.list()

bing_connection_id = ""

async for c in connections:
    if c.name == os.environ["BING_GROUNDING_CONNECTION_NAME"]:
        print(f"Bing connection: {c}\n")
        bing_connection_id = c.id

Bing connection: {'name': 'groundingwithbingsearch', 'id': '/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch', 'type': 'ApiKey', 'target': 'https://api.bing.microsoft.com/', 'isDefault': True, 'credentials': {'type': 'ApiKey'}, 'metadata': {'type': 'bing_grounding', 'ApiType': 'Azure', 'ResourceId': '/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/mmcognitivegrp/providers/Microsoft.Bing/accounts/groundingwithbingsearch'}}



## Define the YAML specification string

In [5]:
spec = f"""
type: foundry_agent
name: {agent_name}
instructions: {instructions}
description: {description}
model:
  id: {agent_settings.model_deployment_name}
  options:
    temperature: 0.4
tools:
  - id: ProductionLinePlugin.get_lines
    type: function
  - id: ProductionLinePlugin.update_line_status
    type: function
  - type: bing_grounding
    options:
      tool_connections:
        - {bing_connection_id}
"""

print(spec)


type: foundry_agent
name: sk_aifoundry_agent-chocolate-lines
instructions: You are a clever agent that supports the chocolate production lines in Ferrero. You have full access to Internet. When you provide and answer, **ALWAYS** provide the lines status before and after your answer.
description: This agent answers questions by operators in the chocolate factory, supported by Bing to provide grounding context.
model:
  id: gpt-4.1
  options:
    temperature: 0.4
tools:
  - id: ProductionLinePlugin.get_lines
    type: function
  - id: ProductionLinePlugin.update_line_status
    type: function
  - type: bing_grounding
    options:
      tool_connections:
        - /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch



## Create the AzureAI Agent from the YAML spec

In [6]:
from semantic_kernel.agents import AgentRegistry

agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=spec,
    client=project_client,
    plugins=[ProductionLinePlugin()],
    settings=agent_settings,
)

# Interacting with an AzureAIAgent
Interaction with the AzureAIAgent is straightforward. The agent maintains the conversation history automatically using a thread.<br/>
The specifics of the Azure AI Agent thread is abstracted away via the AzureAIAgentThread class, which is an implementation of AgentThread.

In [7]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Please start the line that is less affected by ingredient shortages today.", 
    "Check if the place where Ferrero main factory is located is under risk of power failures. If not, please start Mon Chéri Line", 
    "Check if any major logistics delays are affecting hazelnut deliveries to Italy today. If not, restart the production line for Ferrero Rocher.",
    "Is there any expected heatwave in Alba next week that could impact factory cooling systems? If not, please stop the Rocher line.",
]

try:
    i=0
    for user_input in USER_INPUTS:
        i+=1
        thread: AzureAIAgentThread = None
        print(f"************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        response = await agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

************************************
Message 1 from AuthorRole.USER: 'Please start the line that is less affected by ingredient shortages today.'
Message 1 from AuthorRole.ASSISTANT): 'Status before:
- Rocher Line: stopped
- Mon Chéri Line: stopped
- Kinder Bueno Line: stopped

The Kinder Bueno Line has now started, as it is likely the least affected by cocoa shortages due to its higher use of cream and fillers compared to pure chocolate products【5:2†source】【5:3†source】.

Status after:
- Rocher Line: stopped
- Mon Chéri Line: stopped
- Kinder Bueno Line: running'

************************************
Message 2 from AuthorRole.USER: 'Check if the place where Ferrero main factory is located is under risk of power failures. If not, please start Mon Chéri Line'
Message 2 from AuthorRole.ASSISTANT): 'Line status before check:
- Rocher Line: stopped
- Mon Chéri Line: stopped
- Kinder Bueno Line: running

Based on recent reports, Italy—including the Piedmont region where Ferrero’s main factor

# Teardown

In [8]:
# list and delete all files
file_list = await project_client.agents.files.list()
for file in file_list.data:
    await project_client.agents.files.delete(file_id=file.id)

# project_client.agents.vector_stores.delete(vector_store_id=vector_store.id)

# delete thread
print(f"Thread {thread.id} is being deleted")
await thread.delete()

# delete the agent
print(f"{i}) - Agent {agent.name} ({agent.id}) is being deleted\n")
await project_client.agents.delete_agent(agent_id=agent.id)

Thread thread_iWvcSe32tLV0Bg2UzhGfmwlI is being deleted
4) - Agent sk_aifoundry_agent-chocolate-lines (asst_9ed4kI59AB4xYTUCtX32LwJA) is being deleted

